In [2]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import os

# =====================================================================
# 1. HÀM KIỂM ĐỊNH DIEBOLD-MARIANO (DM TEST)
# =====================================================================
def diebold_mariano_test(y_true, y_pred_1, y_pred_2, h=1):
    """
    y_pred_1: Dự báo của Baseline (Finance Only)
    y_pred_2: Dự báo của Hybrid (Finance + TDA)
    h: Horizon (để tính trễ tự tương quan)
    """
    e1 = y_true - y_pred_1
    e2 = y_true - y_pred_2
    
    # Dùng MSE loss: e1^2 - e2^2
    d = e1**2 - e2**2
    mean_d = np.mean(d)
    
    def autocovariance(xi, k):
        N = len(xi)
        x_mean = np.mean(xi)
        return np.sum((xi[:N-k] - x_mean) * (xi[k:] - x_mean)) / N

    gamma = [autocovariance(d, i) for i in range(h)]
    var_d = gamma[0] + 2 * sum(gamma[1:])
    
    N = len(d)
    DM_stat = mean_d / np.sqrt(var_d / N)
    p_value = 2 * (1 - stats.norm.cdf(abs(DM_stat)))
    
    return DM_stat, p_value

def get_stars(p_val):
    if p_val < 0.01: return "*** (Rất mạnh)"
    elif p_val < 0.05: return "** (Mạnh)"
    elif p_val < 0.10: return "* (Yếu)"
    else: return "Không ý nghĩa"

# =====================================================================
# 2. XỬ LÝ DỮ LIỆU & IN KẾT QUẢ
# =====================================================================
# Điền các horizon ông đã chạy vào mảng này (ví dụ: ['7', '14'])
horizons_to_check = ['1', '7', '14'] 
models = ['xgboost', 'lstm']

print("⚖️ BẮT ĐẦU PHÂN XỬ DIEBOLD-MARIANO TEST ⚖️\n")

for h in horizons_to_check:
    print("="*60)
    print(f"🚀 KẾT QUẢ CHO HORIZON: {h} NGÀY")
    print("="*60)
    
    for model in models:
        # Tạo tên file theo chuẩn đã lưu
        file_fin = f'DB/test.preds_{model}_finance_{h}d.csv'
        file_hyb = f'DB/test.preds_{model}_hybrid_{h}d.csv'
        
        # Kiểm tra xem file có tồn tại không
        if not os.path.exists(file_fin) or not os.path.exists(file_hyb):
            print(f"⚠️ Bỏ qua {model.upper()}: Thiếu file CSV (Cần {file_fin} và {file_hyb})")
            continue
            
        # Đọc dữ liệu
        df_fin = pd.read_csv(file_fin)
        df_hyb = pd.read_csv(file_hyb)
        
        # Kiểm tra Ground Truth (y_true) có khớp nhau không
        if not np.allclose(df_fin['y_true'], df_hyb['y_true']):
            print(f"❌ LỖI NGHIÊM TRỌNG ({model.upper()}): Cột y_true giữa 2 file không khớp nhau!")
            continue
            
        y_true = df_fin['y_true'].values
        
        # Chú ý lấy đúng tên cột (dựa vào việc ông đã đổi tên cột ở file export chưa)
        # Nếu chỉ có cột y_pred_..., ta lấy cột thứ 2 (index 2) làm dự báo
        col_pred_fin = df_fin.columns[-1] 
        col_pred_hyb = df_hyb.columns[-1]
        
        y_pred_fin = df_fin[col_pred_fin].values
        y_pred_hyb = df_hyb[col_pred_hyb].values
        
        # Chạy kiểm định DM
        # Truyền h (horizon) bằng số nguyên vào hàm
        dm_stat, p_val = diebold_mariano_test(y_true, y_pred_fin, y_pred_hyb, h=int(h))
        
        # In ra màn hình
        print(f"🔹 {model.upper()} (Finance vs Hybrid):")
        print(f"   - Số lượng Test (N): {len(y_true)}")
        print(f"   - DM Statistic:      {dm_stat:.4f}")
        print(f"   - P-value:           {p_val:.6f} -> {get_stars(p_val)}")
        
        if dm_stat > 0 and p_val < 0.05:
            print(f"   ✅ KẾT LUẬN: Hybrid TDA VƯỢT TRỘI ({model.upper()} ăn tiền!)")
        elif dm_stat < 0:
            print(f"   ❌ KẾT LUẬN: Baseline Finance tốt hơn (DM Stat âm).")
        else:
            print(f"   ⚠️ KẾT LUẬN: Cải thiện chưa đủ ý nghĩa thống kê.")
        print("-" * 40)

⚖️ BẮT ĐẦU PHÂN XỬ DIEBOLD-MARIANO TEST ⚖️

🚀 KẾT QUẢ CHO HORIZON: 1 NGÀY
🔹 XGBOOST (Finance vs Hybrid):
   - Số lượng Test (N): 442
   - DM Statistic:      -1.8292
   - P-value:           0.067371 -> * (Yếu)
   ❌ KẾT LUẬN: Baseline Finance tốt hơn (DM Stat âm).
----------------------------------------
🔹 LSTM (Finance vs Hybrid):
   - Số lượng Test (N): 441
   - DM Statistic:      5.3639
   - P-value:           0.000000 -> *** (Rất mạnh)
   ✅ KẾT LUẬN: Hybrid TDA VƯỢT TRỘI (LSTM ăn tiền!)
----------------------------------------
🚀 KẾT QUẢ CHO HORIZON: 7 NGÀY
🔹 XGBOOST (Finance vs Hybrid):
   - Số lượng Test (N): 442
   - DM Statistic:      0.5753
   - P-value:           0.565065 -> Không ý nghĩa
   ⚠️ KẾT LUẬN: Cải thiện chưa đủ ý nghĩa thống kê.
----------------------------------------
🔹 LSTM (Finance vs Hybrid):
   - Số lượng Test (N): 441
   - DM Statistic:      1.9903
   - P-value:           0.046561 -> ** (Mạnh)
   ✅ KẾT LUẬN: Hybrid TDA VƯỢT TRỘI (LSTM ăn tiền!)
-----------------